# 01 - Làm sạch và chuẩn hóa dữ liệu

Notebook này thực hiện stage đầu tiên: đọc raw data, kiểm tra thông tin, chuẩn hóa kiểu dữ liệu, làm sạch 3 nguồn và tạo `sales_final` trong `data/clean`.

In [1]:
import site
import sys
from pathlib import Path

USER_SITE = site.getusersitepackages()
if Path(USER_SITE).exists() and USER_SITE not in sys.path:
    sys.path.insert(0, USER_SITE)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

from io import StringIO
from src.config import CLEAN_DIR, STAGING_DIR, get_input_files
from src.extract import load_historical_excel, load_2026_excel, load_cost_excel
from src.pipeline import run_cleaning_stage

files = get_input_files()
files

{'historical': WindowsPath('C:/Users/quang/OneDrive/Desktop/DuAnTotNghiep/data/raw/Data_2023-2025.xlsx'),
 'sales_2026': WindowsPath('C:/Users/quang/OneDrive/Desktop/DuAnTotNghiep/data/raw/Data_Sales du an.xlsx'),
 'cost_2026': WindowsPath('C:/Users/quang/OneDrive/Desktop/DuAnTotNghiep/data/raw/Giá vốn theo sp.xlsx')}

In [2]:
raw_hist = load_historical_excel(files["historical"])
raw_2026 = load_2026_excel(files["sales_2026"])
raw_cost = load_cost_excel(files["cost_2026"])

print("Kich thuoc raw 2023-2025:", raw_hist.shape)
print("Kich thuoc raw 2026:", raw_2026.shape)
print("Kich thuoc raw gia von:", raw_cost.shape)
display(raw_hist.head())
display(raw_2026.head())
display(raw_cost.head())

Kich thuoc raw 2023-2025: (34207, 19)
Kich thuoc raw 2026: (20843, 22)
Kich thuoc raw gia von: (242, 6)


,Ngày chứng từ,Số chứng từ,ĐVT,Quy đổi,Số lượng (KG ),Số lượng,Giá bán,Doanh số,Chiết khấu,Doanh thu thuần,Tên nhóm KH1,Tháng,Mã khách hàng,Mã sản phẩm,Kênh,Nhóm sp,Trả hàng,Lợi nhuận,Nguồn sheet
0,2023-01-01,202300000277,TUI,0.20,0.20,1.00,"127,000.00","127,000.00",0,"127,000.00",Nhóm khách hàng nội đia,1,O_CUS_0008,O_PRO_0002,Online,Hòa tan,No,"69,285.00",Data_2023
1,2023-01-01,202300000393,KG,1.00,14.50,14.50,"181,000.00","2,624,500.00",0,"2,624,500.00",Nhóm khách hàng nội đia,1,O_CUS_0095,O_PRO_0009,Khác,Hòa tan,No,"529,094.00",Data_2023
2,2023-01-01,202300000843,HOP,0.30,0.90,3.00,"59,500.00","178,500.00",0,"178,500.00",Nhóm khách hàng nội đia,1,O_CUS_0006,O_PRO_0006,Online,Hòa tan,No,"80,805.00",Data_2023
3,2023-01-01,202300002038,HOP,0.18,1.08,6.00,"47,600.00","285,600.00",13335,"272,265.00",Nhóm khách hàng nội đia,1,O_CUS_0006,O_PRO_0025,Online,Hòa tan,No,"139,386.00",Data_2023
4,2023-01-01,202300004095,TUI,1.60,12.80,8.00,"251,000.00","2,008,000.00",0,"2,008,000.00",Nhóm khách hàng nội đia,1,O_CUS_0149,O_PRO_0001,Khác,Hòa tan,No,"1,103,403.00",Data_2023


,Ngày chứng từ,Số chứng từ,ĐVT,Tổng số lượng bán,Đơn Giá Chuẩn,Doanh số bán,Chiết khấu,Doanh thu thuần,Tổng số lượng trả lại,Giá trị trả lại,Kênh,Tỷ lệ chuyển đổi,Chi nhánh,Vùng doanh thu,Khối lượng,Mã sản phẩm,Tên sản phẩm,Nhóm sản phẩm,Loại sản phẩm,Mã khách hàng2,Tên khách hàng2,Nhóm khách hàng
0,2026-01-01,2501007972,Ly,1,"21,000.00","21,000.00","7,600.00","13,400.00",NaN,NaN,Quán nội bộ,0.00,Quán Tây Nguyên 1,Miền Trung - Tây Nguyên,0.00,PROD_00182,Coffee Beverage 039,Đồ Uống,Thức uống cà phê,CUST_00031,Internal Customer 001,Internal
1,2026-01-23,BH00599.26LM,Bịch,24,"52,400.00","1,120,000.00",0.00,"1,037,037.04",NaN,NaN,HORECA,NaN,Công ty,Miền Bắc,24.00,PROD_00040,Ground Coffee 025,Rang Xay,Cà phê rang xay,CUST_00097,F&B Customer 068,F&B
2,2026-01-01,2601000001,Ly,3,"21,000.00","63,000.00",0.00,"63,000.00",NaN,NaN,Quán nội bộ,0.00,Quán Tây Nguyên 1,Miền Trung - Tây Nguyên,0.00,PROD_00182,Coffee Beverage 039,Đồ Uống,Thức uống cà phê,CUST_00031,Internal Customer 001,Internal
3,2026-01-01,2601000001,Ly,2,"26,000.00","52,000.00",0.00,"52,000.00",NaN,NaN,Quán nội bộ,0.00,Quán Tây Nguyên 1,Miền Trung - Tây Nguyên,0.00,PROD_00181,Coffee Beverage 038,Đồ Uống,Thức uống cà phê,CUST_00031,Internal Customer 001,Internal
4,2026-01-23,BH00605.26LM,Bịch,1,"58,000.00","51,852.00",0.00,"48,011.11",NaN,NaN,Online,NaN,Công ty,Online,1.00,PROD_00040,Ground Coffee 025,Rang Xay,Cà phê rang xay,CUST_00004,Online Customer 004,Online


,Mã Sản phẩm,Nhóm SP Chuẩn,Loại SP Chuẩn,Tên SP Mới,Giá bán lẻ,Giá vốn
0,PROD_00001,Hòa Tan,Cà phê hòa tan,Instant Coffee 001,"121,000.00",49041
1,PROD_00002,Hòa Tan,Cà phê hòa tan,Instant Coffee 002,"240,150.00",124470
2,PROD_00003,Hòa Tan,Cà phê hòa tan,Instant Coffee 003,"30,350.00",14386
3,PROD_00004,Hòa Tan,Cà phê hòa tan,Instant Coffee 004,"52,800.00",27583
4,PROD_00005,Hòa Tan,Cà phê hòa tan,Instant Coffee 005,"68,500.00",28448


In [3]:
# Kiểm tra info và describe trước khi làm sạch
for name, df in {"raw_hist": raw_hist, "raw_2026": raw_2026, "raw_cost": raw_cost}.items():
    print("\n=====", name, "=====")
    buffer = StringIO()
    df.info(buf=buffer)
    print(buffer.getvalue())
    display(df.describe(include="all"))


===== raw_hist =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34207 entries, 0 to 34206
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Ngày chứng từ    34207 non-null  datetime64[ns]
 1   Số chứng từ      34207 non-null  object        
 2   ĐVT              34207 non-null  object        
 3   Quy đổi          34207 non-null  float64       
 4   Số lượng (KG )   34207 non-null  float64       
 5   Số lượng         34207 non-null  float64       
 6   Giá bán          34207 non-null  float64       
 7   Doanh số         34207 non-null  float64       
 8   Chiết khấu       34207 non-null  int64         
 9   Doanh thu thuần  34207 non-null  float64       
 10  Tên nhóm KH1     34207 non-null  object        
 11  Tháng            34207 non-null  int64         
 12  Mã khách hàng    34207 non-null  object        
 13  Mã sản phẩm      34207 non-null  object        
 14  Kênh            

,Ngày chứng từ,Số chứng từ,ĐVT,Quy đổi,Số lượng (KG ),Số lượng,Giá bán,Doanh số,Chiết khấu,Doanh thu thuần,Tên nhóm KH1,Tháng,Mã khách hàng,Mã sản phẩm,Kênh,Nhóm sp,Trả hàng,Lợi nhuận,Nguồn sheet
count,34207,34207,34207,"34,207.00","34,207.00","34,207.00","34,207.00","34,207.00","34,207.00","34,207.00",34207,"34,207.00",34207,34207,34207,34207,34207,"34,207.00",34207
unique,NaN,22041,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,225,95,5,2,2,NaN,3
top,NaN,GL250905.015,KG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nhóm khách hàng nội đia,NaN,O_CUS_0006,O_PRO_0011,Online,Rang xay,No,NaN,Data_2025
freq,NaN,35,16926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33910,NaN,10861,3865,15051,18162,33357,NaN,13211
mean,2024-08-13 21:41:30.121904896,NaN,NaN,0.69,24.28,112.10,"178,872.31","3,476,784.35",844.21,"3,475,940.13",NaN,6.98,NaN,NaN,NaN,NaN,NaN,"815,094.41",NaN
min,2023-01-01 00:00:00,NaN,NaN,0.00,-288.00,"-1,800.00",0.00,"-48,600,000.00",0.00,"-48,600,000.00",NaN,1.00,NaN,NaN,NaN,NaN,NaN,"-24,744,153.00",NaN
25%,2023-11-12 00:00:00,NaN,NaN,0.20,0.50,1.00,"59,500.00","136,000.00",0.00,"138,000.00",NaN,4.00,NaN,NaN,NaN,NaN,NaN,"49,369.00",NaN
50%,2024-09-09 00:00:00,NaN,NaN,1.00,1.50,3.00,"170,000.00","366,000.00",0.00,"400,000.00",NaN,7.00,NaN,NaN,NaN,NaN,NaN,"136,374.00",NaN
75%,2025-05-29 00:00:00,NaN,NaN,1.00,8.00,10.00,"221,000.00","1,547,000.00",0.00,"1,636,800.00",NaN,10.00,NaN,NaN,NaN,NaN,NaN,"432,929.00",NaN
max,2025-12-31 00:00:00,NaN,NaN,5.95,"15,427.50","525,000.00","1,790,000.00","4,985,772,000.00","1,354,000.00","2,345,542,927.70",NaN,12.00,NaN,NaN,NaN,NaN,NaN,"395,789,082.00",NaN



===== raw_2026 =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20843 entries, 0 to 20842
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Ngày chứng từ          20843 non-null  datetime64[ns]
 1   Số chứng từ            20843 non-null  object        
 2   ĐVT                    20843 non-null  object        
 3   Tổng số lượng bán      20843 non-null  int64         
 4   Đơn Giá Chuẩn          20843 non-null  float64       
 5   Doanh số bán           20843 non-null  float64       
 6   Chiết khấu             19581 non-null  float64       
 7   Doanh thu thuần        20843 non-null  float64       
 8   Tổng số lượng trả lại  3222 non-null   float64       
 9   Giá trị trả lại        3222 non-null   float64       
 10  Kênh                   20843 non-null  object        
 11  Tỷ lệ chuyển đổi       18278 non-null  float64       
 12  Chi nhánh              20843 non-null 

,Ngày chứng từ,Số chứng từ,ĐVT,Tổng số lượng bán,Đơn Giá Chuẩn,Doanh số bán,Chiết khấu,Doanh thu thuần,Tổng số lượng trả lại,Giá trị trả lại,Kênh,Tỷ lệ chuyển đổi,Chi nhánh,Vùng doanh thu,Khối lượng,Mã sản phẩm,Tên sản phẩm,Nhóm sản phẩm,Loại sản phẩm,Mã khách hàng2,Tên khách hàng2,Nhóm khách hàng
count,20843,20843,20843,"20,843.00","20,843.00","20,843.00","19,581.00","20,843.00","3,222.00","3,222.00",20843,"18,278.00",20843,20843,"20,437.00",20843,20843,20843,20843,20843,20843,20843
unique,NaN,13273,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,NaN,4,8,NaN,242,242,4,11,155,155,9
top,NaN,BH03892.26LM,Ly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Quán nội bộ,NaN,Công ty,Miền Trung - Tây Nguyên,NaN,PROD_00182,Coffee Beverage 039,Đồ Uống,Thức uống cà phê,CUST_00032,Internal Customer 002,Internal
freq,NaN,21,10957,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14688,NaN,6155,11996,NaN,2520,2520,11396,8374,5797,5797,14688
mean,2026-03-17 02:45:07.211054080,NaN,NaN,22.45,"109,453.39","918,028.63","13,699.11","898,516.64",0.20,"21,001.57",NaN,0.25,NaN,NaN,19.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2026-01-01 00:00:00,NaN,NaN,0.00,"-37,667.00","-1,101,861.00",0.00,"-1,020,241.67",0.00,0.00,NaN,0.00,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2026-02-05 00:00:00,NaN,NaN,1.00,"26,000.00","32,000.00",0.00,"31,454.55",0.00,0.00,NaN,0.00,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2026-03-18 00:00:00,NaN,NaN,1.00,"47,000.00","62,000.00",0.00,"52,000.00",0.00,0.00,NaN,0.00,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2026-04-23 00:00:00,NaN,NaN,2.00,"147,000.00","250,000.00",0.00,"240,741.00",0.00,0.00,NaN,0.25,NaN,NaN,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,2026-05-28 00:00:00,NaN,NaN,"123,000.00","246,751,999.76","2,582,486,400.00","4,166,667.00","2,582,486,400.00",96.00,"7,333,333.00",NaN,5.95,NaN,NaN,"123,000.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN



===== raw_cost =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242 entries, 0 to 241
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Mã Sản phẩm    242 non-null    object 
 1   Nhóm SP Chuẩn  242 non-null    object 
 2   Loại SP Chuẩn  242 non-null    object 
 3   Tên SP Mới     242 non-null    object 
 4   Giá bán lẻ     242 non-null    float64
 5   Giá vốn        242 non-null    int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 11.5+ KB



,Mã Sản phẩm,Nhóm SP Chuẩn,Loại SP Chuẩn,Tên SP Mới,Giá bán lẻ,Giá vốn
count,242,242,242,242,242.00,242.00
unique,242,4,11,242,NaN,NaN
top,PROD_00001,Đồ Uống,Thức uống cà phê,Instant Coffee 001,NaN,NaN
freq,1,146,60,1,NaN,NaN
mean,NaN,NaN,NaN,NaN,"123,348.14","90,992.79"
std,NaN,NaN,NaN,NaN,"242,035.65","186,323.73"
min,NaN,NaN,NaN,NaN,"2,000.00",994.00
25%,NaN,NaN,NaN,NaN,"42,000.00","28,405.00"
50%,NaN,NaN,NaN,NaN,"51,000.00","36,702.50"
75%,NaN,NaN,NaN,NaN,"95,882.11","62,742.00"


In [4]:
# Chạy stage làm sạch: fill null cột số = 0, chuẩn hóa text, map nhãn chuẩn, merge giá vốn và tạo lại lợi nhuận 2026
result = run_cleaning_stage()
hist_clean = result["hist_clean"]
sales_2026_clean = result["sales_2026_clean"]
cost_clean = result["cost_clean"]
final_sales = result["final_sales"]
quality = result["data_quality_overview"]

print("Da xuat staging vao:", STAGING_DIR)
print("Da xuat clean vao:", CLEAN_DIR)
display(quality)
display(final_sales.head())

Da xuat staging vao: C:\Users\quang\OneDrive\Desktop\DuAnTotNghiep\data\staging
Da xuat clean vao: C:\Users\quang\OneDrive\Desktop\DuAnTotNghiep\data\clean


,Bảng,Số dòng,Số cột,Số ô null,Số dòng duplicate
0,raw_sales_2023_2025,34207,19,0,29
1,raw_sales_2026,20843,22,39475,14
2,raw_cost_2026,242,6,0,0
3,sales_2023_2025_clean,34207,29,68414,3
4,sales_2026_clean,20843,39,0,0
5,cost_2026_clean,242,6,0,0
6,sales_final,55050,38,5539,3


,Ngày chứng từ,Năm,Tháng,Quý,Năm-Tháng,Số chứng từ,ĐVT,Số lượng,Khối lượng KG,Đơn giá,Doanh số,Chiết khấu,Doanh thu thuần,Số lượng trả lại,Giá trị trả lại,Giá vốn đơn vị,Tổng giá vốn,Lợi nhuận,Lợi nhuận tính lại,Mã khách hàng,Tên khách hàng,Nhóm khách hàng,Mã sản phẩm,Tên sản phẩm,Nhóm sản phẩm,Loại sản phẩm,Kênh,Chi nhánh,Vùng doanh thu,Trả hàng,Giai đoạn,Nguồn dữ liệu,Có dữ liệu giá vốn,Cờ duplicate,Biên lợi nhuận,Tỷ lệ chiết khấu,Tỷ lệ hoàn trả,Giá trị TB/dòng
0,2023-01-01,2023,1,1,2023-01,202300000277,TUI,1.00,0.20,"127,000.00","127,000.00",0.00,"127,000.00",0.00,0.00,"57,715.00","57,715.00","69,285.00","69,285.00",O_CUS_0008,Không có dữ liệu,Nhóm khách hàng nội địa,O_PRO_0002,Không có dữ liệu,Hòa tan,Không có dữ liệu,Online,Không áp dụng,Không áp dụng,Không,Trước tái cấu trúc,2023-2025,False,False,0.55,0.00,0.00,"127,000.00"
1,2023-01-01,2023,1,1,2023-01,202300000393,KG,14.50,14.50,"181,000.00","2,624,500.00",0.00,"2,624,500.00",0.00,0.00,"144,510.76","2,095,406.00","529,094.00","529,094.00",O_CUS_0095,Không có dữ liệu,Nhóm khách hàng nội địa,O_PRO_0009,Không có dữ liệu,Hòa tan,Không có dữ liệu,Khác,Không áp dụng,Không áp dụng,Không,Trước tái cấu trúc,2023-2025,False,False,0.20,0.00,0.00,"2,624,500.00"
2,2023-01-01,2023,1,1,2023-01,202300000843,HOP,3.00,0.90,"59,500.00","178,500.00",0.00,"178,500.00",0.00,0.00,"32,565.00","97,695.00","80,805.00","80,805.00",O_CUS_0006,Không có dữ liệu,Nhóm khách hàng nội địa,O_PRO_0006,Không có dữ liệu,Hòa tan,Không có dữ liệu,Online,Không áp dụng,Không áp dụng,Không,Trước tái cấu trúc,2023-2025,False,False,0.45,0.00,0.00,"178,500.00"
3,2023-01-01,2023,1,1,2023-01,202300002038,HOP,6.00,1.08,"47,600.00","285,600.00","13,335.00","272,265.00",0.00,0.00,"22,146.50","132,879.00","139,386.00","139,386.00",O_CUS_0006,Không có dữ liệu,Nhóm khách hàng nội địa,O_PRO_0025,Không có dữ liệu,Hòa tan,Không có dữ liệu,Online,Không áp dụng,Không áp dụng,Không,Trước tái cấu trúc,2023-2025,False,False,0.51,0.05,0.00,"272,265.00"
4,2023-01-01,2023,1,1,2023-01,202300004095,TUI,8.00,12.80,"251,000.00","2,008,000.00",0.00,"2,008,000.00",0.00,0.00,"113,074.62","904,597.00","1,103,403.00","1,103,403.00",O_CUS_0149,Không có dữ liệu,Nhóm khách hàng nội địa,O_PRO_0001,Không có dữ liệu,Hòa tan,Không có dữ liệu,Khác,Không áp dụng,Không áp dụng,Không,Trước tái cấu trúc,2023-2025,False,False,0.55,0.00,0.00,"2,008,000.00"


In [5]:
# Show duplicate nh?ng kh?ng x?a b?
print("Duplicate 2023-2025:", int(hist_clean.duplicated().sum()))
print("Duplicate 2026:", int(sales_2026_clean.duplicated().sum()))
print("Duplicate final:", int(final_sales.duplicated().sum()))

# Ki?m tra merge gi? v?n 2026
print("Ty le dong 2026 match gia von:", sales_2026_clean["Có dữ liệu giá vốn"].mean())


Duplicate 2023-2025: 3
Duplicate 2026: 0
Duplicate final: 3
Ty le dong 2026 match gia von: 1.0
